# Week 2 – Data Collection & Preprocessing  
## Predictive Maintenance for IT Infrastructure  

In this notebook, we focus on preparing a clean and structured dataset for predicting server failures.

We are using a server log dataset obtained from Kaggle, which contains system-level activity and log information. These logs help us understand system behavior and identify patterns that may lead to failures.

The goal of this notebook is to:
- Load and explore the dataset  
- Clean and preprocess the data  
- Prepare it for machine learning models  

In [1]:
# Basic libraries
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

## Dataset Loading

We load the dataset directly from Kaggle.
Make sure you have uploaded your Kaggle API key (kaggle.json).

In [2]:
# Install Kaggle
!pip install kaggle

# Upload kaggle.json manually in Colab
from google.colab import files
files.upload()

# Setup Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d radioactivem/server-logs-data-for-server-failure

# Unzip
!unzip server-logs-data-for-server-failure.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/radioactivem/server-logs-data-for-server-failure
License(s): ODC Public Domain Dedication and Licence (PDDL)
100% 958k/958k [00:00<00:00, 85.0MB/s]

Archive:  server-logs-data-for-server-failure.zip
  inflating: Modified_Simulated_Banking_Server_Data.csv  
  inflating: Simulated_Banking_Server_Data.csv  


## Reading the Dataset

In [4]:
# Load dataset (update filename if needed)
df = pd.read_csv('Modified_Simulated_Banking_Server_Data.csv')

# Preview
print("Original dataset loaded:")
df.head()

Original dataset loaded:


,Timestamp,CPU_Usage(%),Memory_Usage(%),Disk_IO(MB/s),Network_Latency(ms),Error_Rate,Downtime
0,2023-01-01 00:00:00,57.697437,46.442143,172.495710,12.253697,0.013118,0
1,2023-01-01 01:00:00,47.945152,53.724506,168.189358,26.964343,0.039280,1
2,2023-01-01 02:00:00,60.134829,47.990294,96.729926,36.358097,0.016947,0
3,2023-01-01 03:00:00,75.165068,62.337067,188.239521,53.808925,0.021474,1
4,2023-01-01 04:00:00,46.542527,87.290604,70.191911,78.750784,0.033022,0


## Understanding the Dataset

We examine:
- Number of rows and columns  
- Data types  
- Missing values  

In [5]:
# Shape
print("Shape:", df.shape)

# Info
df.info()

# Missing values
df.isnull().sum()

Shape: (10000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Timestamp            10000 non-null  object 
 1   CPU_Usage(%)         10000 non-null  float64
 2   Memory_Usage(%)      10000 non-null  float64
 3   Disk_IO(MB/s)        10000 non-null  float64
 4   Network_Latency(ms)  10000 non-null  float64
 5   Error_Rate           10000 non-null  float64
 6   Downtime             10000 non-null  int64  
dtypes: float64(5), int64(1), object(1)
memory usage: 547.0+ KB


,0
Timestamp,0
CPU_Usage(%),0
Memory_Usage(%),0
Disk_IO(MB/s),0
Network_Latency(ms),0
Error_Rate,0
Downtime,0


## Data Cleaning

Steps performed:
- Handling missing values  
- Removing duplicates  
- Fixing data types  

In [6]:
# Remove duplicates
df.drop_duplicates(inplace=True)

# Handle missing values
df.fillna(df.mean(numeric_only=True), inplace=True)

# Convert timestamp if present
if "Timestamp" in df.columns:
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors='coerce')

df.isnull().sum()

,0
Timestamp,0
CPU_Usage(%),0
Memory_Usage(%),0
Disk_IO(MB/s),0
Network_Latency(ms),0
Error_Rate,0
Downtime,0


In [9]:
# Create the 'failure' target variable based on 'Downtime'
df["failure"] = (df["Downtime"] > 0).astype(int)

# Verify creation by printing value counts
print("Value counts for the new 'failure' column:")
display(df["failure"].value_counts())

Value counts for the new 'failure' column:


,count
failure,
0,5189
1,4811


## Encoding Categorical Variables

Machine learning models require numerical inputs, so we convert categorical data into numeric form.

In [10]:
le = LabelEncoder()

for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col])

## Feature Scaling

We normalize the data so that all features are on the same scale.
This is especially important for deep learning models.

In [11]:
scaler = StandardScaler()

# Exclude target variable
target = "failure"

features = df.drop(columns=[target])

# Do NOT scale timestamp
if "Timestamp" in features.columns:
    features = features.drop(columns=["Timestamp"])

scaled_features = scaler.fit_transform(features)

X = pd.DataFrame(scaled_features, columns=features.columns)
y = df[target]

## Final Prepared Dataset

The dataset is now:
- Cleaned  
- Encoded  
- Scaled  
- Ready for model training  

In [13]:
final_df = pd.concat([X, y], axis=1)

final_df.to_csv("clean_dataset.csv", index=False)

final_df.head()

,CPU_Usage(%),Memory_Usage(%),Disk_IO(MB/s),Network_Latency(ms),Error_Rate,Downtime,failure
0,0.444708,-0.727393,0.278273,-1.596836,-0.608875,-0.962888,0
1,-0.200414,-0.381596,0.209169,-1.018029,0.983056,1.038542,1
2,0.605943,-0.653880,-0.937548,-0.648421,-0.375905,-0.962888,0
3,1.600206,0.027366,0.530915,0.038202,-0.100446,1.038542,1
4,-0.293198,1.212266,-1.363406,1.019568,0.602275,-0.962888,0
